In [3]:
# =========================
# INCISO 1
# Descargar dataset de Kaggle, dividirlo en 70/15/15
# e implementar Data Augmentation
# =========================

# Si kagglehub no está instalado, descomenta esta línea:
!pip -q install kagglehub

import os
import random
import numpy as np
from pathlib import Path
from collections import Counter
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

import kagglehub

# -------------------------
# 1. Reproducibilidad
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -------------------------
# 2. Descargar dataset desde Kaggle
# -------------------------
dataset_root = kagglehub.dataset_download("aryashah2k/mango-leaf-disease-dataset")
print("Path descargado por kagglehub:", dataset_root)

dataset_root = Path(dataset_root)

# -------------------------
# 3. Encontrar automáticamente la carpeta que contiene las clases
# -------------------------
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images_recursive(folder):
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in valid_exts)

def get_immediate_class_dirs(folder):
    subdirs = [d for d in folder.iterdir() if d.is_dir()]
    valid_dirs = []
    for d in subdirs:
        if count_images_recursive(d) > 0:
            valid_dirs.append(d)
    return sorted(valid_dirs, key=lambda x: x.name.lower())

candidates = []

# revisa la carpeta raíz descargada y subcarpetas
for d in [dataset_root] + [p for p in dataset_root.rglob("*") if p.is_dir()]:
    try:
        class_dirs_candidate = get_immediate_class_dirs(d)
        if len(class_dirs_candidate) >= 2:
            total_imgs = sum(count_images_recursive(cd) for cd in class_dirs_candidate)
            candidates.append((d, len(class_dirs_candidate), total_imgs))
    except:
        pass

if len(candidates) == 0:
    raise ValueError(
        "No se pudo encontrar automáticamente una carpeta con subcarpetas de clases."
    )

# elegir mejor candidata
candidates = sorted(candidates, key=lambda x: (x[1], x[2]), reverse=True)
DATASET_DIR = candidates[0][0]

print("\nCandidatos detectados:")
for c in candidates[:10]:
    print(f"- {c[0]} | clases: {c[1]} | imágenes: {c[2]}")

print("\nUsando carpeta del dataset:", DATASET_DIR)

# -------------------------
# 4. Detectar clases
# -------------------------
class_dirs = get_immediate_class_dirs(DATASET_DIR)

if len(class_dirs) == 0:
    raise ValueError("No se encontraron carpetas de clases en el dataset.")

class_names = [d.name for d in class_dirs]
class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}

print("\nClases encontradas:")
for cls_dir in class_dirs:
    n = count_images_recursive(cls_dir)
    print(f"- {cls_dir.name}: {n} imágenes")

print("Cantidad de clases encontradas:", len(class_names))

if len(class_names) != 8:
    print(f"\nADVERTENCIA: se encontraron {len(class_names)} clases, no 8.")
    print("Revisa si este dataset tiene una estructura distinta a la esperada.\n")

# -------------------------
# 5. Recolectar imágenes y labels
# -------------------------
filepaths = []
labels = []

for cls_name, cls_dir in zip(class_names, class_dirs):
    cls_idx = class_to_idx[cls_name]
    for p in cls_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in valid_exts:
            filepaths.append(str(p))
            labels.append(cls_idx)

if len(filepaths) == 0:
    raise ValueError("No se encontraron imágenes válidas en el dataset.")

filepaths = np.array(filepaths)
labels = np.array(labels)

print("Total de imágenes encontradas:", len(filepaths))

print("\nDistribución por clase:")
counts = Counter(labels)
for cls_name in class_names:
    idx = class_to_idx[cls_name]
    print(f"- {cls_name}: {counts[idx]}")

# -------------------------
# 6. Split 70/15/15 estratificado
# -------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    filepaths,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)

print("\nTamaños de particiones:")
print(f"Entrenamiento: {len(X_train)} imágenes ({len(X_train)/len(filepaths):.2%})")
print(f"Validación:    {len(X_val)} imágenes ({len(X_val)/len(filepaths):.2%})")
print(f"Prueba:        {len(X_test)} imágenes ({len(X_test)/len(filepaths):.2%})")

# -------------------------
# 7. Transformaciones
# -------------------------
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# Data augmentation para entrenamiento
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(20),                 # rotaciones
    transforms.RandomHorizontalFlip(p=0.5),        # flip horizontal
    transforms.RandomVerticalFlip(p=0.5),          # flip vertical
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),  # recortes
    transforms.ToTensor(),
])

# Validación y prueba sin augmentation
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# -------------------------
# 8. Dataset personalizado
# -------------------------
class MangoDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        label = int(self.labels[idx])

        if self.transform:
            img = self.transform(img)

        return img, label

train_dataset = MangoDataset(X_train, y_train, transform=train_transform)
val_dataset   = MangoDataset(X_val, y_val, transform=eval_transform)
test_dataset  = MangoDataset(X_test, y_test, transform=eval_transform)

# -------------------------
# 9. DataLoaders
# -------------------------
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("\nDataLoaders creados correctamente.")

# -------------------------
# 10. Verificación rápida
# -------------------------
images, labels_batch = next(iter(train_loader))
print("Batch de imágenes:", images.shape)   # esperado: [batch, 3, 224, 224]
print("Batch de labels:", labels_batch.shape)
print("Mapa de clases:", class_to_idx)
print("Carpeta final usada:", DATASET_DIR)

Device: cuda
Using Colab cache for faster access to the 'mango-leaf-disease-dataset' dataset.
Path descargado por kagglehub: /kaggle/input/mango-leaf-disease-dataset

Candidatos detectados:
- /kaggle/input/mango-leaf-disease-dataset | clases: 8 | imágenes: 4000

Usando carpeta del dataset: /kaggle/input/mango-leaf-disease-dataset

Clases encontradas:
- Anthracnose: 500 imágenes
- Bacterial Canker: 500 imágenes
- Cutting Weevil: 500 imágenes
- Die Back: 500 imágenes
- Gall Midge: 500 imágenes
- Healthy: 500 imágenes
- Powdery Mildew: 500 imágenes
- Sooty Mould: 500 imágenes
Cantidad de clases encontradas: 8
Total de imágenes encontradas: 4000

Distribución por clase:
- Anthracnose: 500
- Bacterial Canker: 500
- Cutting Weevil: 500
- Die Back: 500
- Gall Midge: 500
- Healthy: 500
- Powdery Mildew: 500
- Sooty Mould: 500

Tamaños de particiones:
Entrenamiento: 2800 imágenes (70.00%)
Validación:    600 imágenes (15.00%)
Prueba:        600 imágenes (15.00%)

DataLoaders creados correctament

In [4]:
# =========================
# INCISO 2.a
# ResNet50 preentrenado en ImageNet
# Congelar base y reemplazar cabezal para 8 clases
# =========================

import torch
import torch.nn as nn
from torchvision import models

NUM_CLASSES = len(class_names)

resnet_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Congelar capas base
for param in resnet_model.parameters():
    param.requires_grad = False

# Reemplazar capa final
in_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(in_features, NUM_CLASSES)

resnet_model = resnet_model.to(device)

print("Modelo: ResNet50")
print("Nueva capa final:", resnet_model.fc)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 209MB/s]


Modelo: ResNet50
Nueva capa final: Linear(in_features=2048, out_features=8, bias=True)


In [5]:
# =========================
# INCISO 2.b
# InceptionV3 preentrenado en ImageNet
# Congelar base y reemplazar cabezal para 8 clases
# =========================

import torch
import torch.nn as nn
from torchvision import models

NUM_CLASSES = len(class_names)

inception_model = models.inception_v3(
    weights=models.Inception_V3_Weights.IMAGENET1K_V1,
    aux_logits=True
)

# Congelar capas base
for param in inception_model.parameters():
    param.requires_grad = False

# Reemplazar capa final principal
in_features_main = inception_model.fc.in_features
inception_model.fc = nn.Linear(in_features_main, NUM_CLASSES)

# Reemplazar capa auxiliar
if inception_model.AuxLogits is not None:
    in_features_aux = inception_model.AuxLogits.fc.in_features
    inception_model.AuxLogits.fc = nn.Linear(in_features_aux, NUM_CLASSES)

inception_model = inception_model.to(device)

print("Modelo: InceptionV3")
print("Nueva capa final principal:", inception_model.fc)
print("Nueva capa auxiliar:", inception_model.AuxLogits.fc if inception_model.AuxLogits else "No AuxLogits")

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:01<00:00, 79.6MB/s]


Modelo: InceptionV3
Nueva capa final principal: Linear(in_features=2048, out_features=8, bias=True)
Nueva capa auxiliar: Linear(in_features=768, out_features=8, bias=True)


In [6]:
# =========================
# INCISO 2.c
# MobileNetV2 preentrenado en ImageNet
# Congelar base y reemplazar cabezal para 8 clases
# =========================

import torch
import torch.nn as nn
from torchvision import models

NUM_CLASSES = len(class_names)

mobilenet_model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)

# Congelar capas base
for param in mobilenet_model.parameters():
    param.requires_grad = False

# Reemplazar capa final
in_features = mobilenet_model.classifier[1].in_features
mobilenet_model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

mobilenet_model = mobilenet_model.to(device)

print("Modelo: MobileNetV2")
print("Nuevo clasificador:", mobilenet_model.classifier)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 59.4MB/s]

Modelo: MobileNetV2
Nuevo clasificador: Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=8, bias=True)
)


In [8]:
# =========================
# INCISO 3 (CORREGIDO)
# Entrenamiento de los 3 modelos con Cross-Entropy + Adam + Early Stopping
# Corrige InceptionV3 usando imágenes de 299x299
# =========================

import copy
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader

# -------------------------
# 1. Parámetros de entrenamiento
# -------------------------
EPOCHS = 20
LEARNING_RATE = 1e-3
PATIENCE = 4

criterion = nn.CrossEntropyLoss()

# -------------------------
# 2. Crear loaders especiales para InceptionV3 (299x299)
# -------------------------
# Reutilizamos X_train, y_train, X_val, y_val del inciso 1
# y la clase MangoDataset que ya definiste antes

inception_train_transform = transforms.Compose([
    transforms.Resize((340, 340)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomResizedCrop(299, scale=(0.8, 1.0)),
    transforms.ToTensor(),
])

inception_eval_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
])

inception_train_dataset = MangoDataset(X_train, y_train, transform=inception_train_transform)
inception_val_dataset   = MangoDataset(X_val, y_val, transform=inception_eval_transform)

inception_train_loader = DataLoader(
    inception_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

inception_val_loader = DataLoader(
    inception_val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

# -------------------------
# 3. Función de entrenamiento
# -------------------------
def train_one_model(model, model_name, train_loader_used, val_loader_used, epochs=20, lr=1e-3, patience=4):
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    patience_counter = 0
    history = []

    start_total = time.time()

    for epoch in range(epochs):
        # ===== Entrenamiento =====
        model.train()
        running_loss = 0.0
        running_corrects = 0
        total_train = 0

        for inputs, labels in train_loader_used:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            if isinstance(model, models.Inception3):
                outputs, aux_outputs = model(inputs)
                loss_main = criterion(outputs, labels)
                loss_aux = criterion(aux_outputs, labels)
                loss = loss_main + 0.4 * loss_aux
                preds = torch.argmax(outputs, dim=1)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                preds = torch.argmax(outputs, dim=1)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels).item()
            total_train += labels.size(0)

        train_loss = running_loss / total_train
        train_acc = running_corrects / total_train

        # ===== Validación =====
        model.eval()
        val_loss_sum = 0.0
        val_corrects = 0
        total_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader_used:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)

                if isinstance(model, models.Inception3):
                    # En evaluación Inception devuelve solo el output principal
                    if hasattr(outputs, "logits"):
                        outputs = outputs.logits

                loss = criterion(outputs, labels)
                preds = torch.argmax(outputs, dim=1)

                val_loss_sum += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels).item()
                total_val += labels.size(0)

        val_loss = val_loss_sum / total_val
        val_acc = val_corrects / total_val

        history.append({
            "Modelo": model_name,
            "Epoch": epoch + 1,
            "Train Loss": train_loss,
            "Train Acc": train_acc,
            "Val Loss": val_loss,
            "Val Acc": val_acc
        })

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping activado para {model_name}")
                break

    total_time = time.time() - start_total
    model.load_state_dict(best_model_wts)

    return model, pd.DataFrame(history), total_time

# -------------------------
# 4. Entrenar ResNet50
# -------------------------
resnet_model, resnet_history, resnet_train_time = train_one_model(
    resnet_model,
    "ResNet50",
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    patience=PATIENCE
)

# -------------------------
# 5. Entrenar InceptionV3 con loaders 299x299
# -------------------------
inception_model, inception_history, inception_train_time = train_one_model(
    inception_model,
    "InceptionV3",
    inception_train_loader,
    inception_val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    patience=PATIENCE
)

# -------------------------
# 6. Entrenar MobileNetV2
# -------------------------
mobilenet_model, mobilenet_history, mobilenet_train_time = train_one_model(
    mobilenet_model,
    "MobileNetV2",
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    patience=PATIENCE
)

# -------------------------
# 7. Unir historiales
# -------------------------
history_df = pd.concat(
    [resnet_history, inception_history, mobilenet_history],
    ignore_index=True
)

print("\nEntrenamiento finalizado.")
print("\nTiempo total de entrenamiento:")
print(f"ResNet50:    {resnet_train_time/60:.2f} min")
print(f"InceptionV3: {inception_train_time/60:.2f} min")
print(f"MobileNetV2: {mobilenet_train_time/60:.2f} min")

[ResNet50] Epoch 1/20 | Train Loss: 0.0302 | Train Acc: 0.9957 | Val Loss: 0.0194 | Val Acc: 1.0000
[ResNet50] Epoch 2/20 | Train Loss: 0.0277 | Train Acc: 0.9954 | Val Loss: 0.0264 | Val Acc: 0.9950
[ResNet50] Epoch 3/20 | Train Loss: 0.0226 | Train Acc: 0.9971 | Val Loss: 0.0202 | Val Acc: 0.9983
[ResNet50] Epoch 4/20 | Train Loss: 0.0203 | Train Acc: 0.9971 | Val Loss: 0.0210 | Val Acc: 0.9950
[ResNet50] Epoch 5/20 | Train Loss: 0.0202 | Train Acc: 0.9975 | Val Loss: 0.0148 | Val Acc: 0.9983
[ResNet50] Epoch 6/20 | Train Loss: 0.0190 | Train Acc: 0.9975 | Val Loss: 0.0146 | Val Acc: 0.9983
[ResNet50] Epoch 7/20 | Train Loss: 0.0233 | Train Acc: 0.9939 | Val Loss: 0.0179 | Val Acc: 0.9983
[ResNet50] Epoch 8/20 | Train Loss: 0.0184 | Train Acc: 0.9971 | Val Loss: 0.0187 | Val Acc: 0.9983
[ResNet50] Epoch 9/20 | Train Loss: 0.0152 | Train Acc: 0.9975 | Val Loss: 0.0136 | Val Acc: 0.9983
[ResNet50] Epoch 10/20 | Train Loss: 0.0142 | Train Acc: 0.9982 | Val Loss: 0.0109 | Val Acc: 0.9983

In [9]:
# =========================
# INCISO 4
# Métricas finales:
# Accuracy, F1 Macro, tamaño del modelo y tiempo de inferencia
# =========================

import os
import time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from torchvision import models

def evaluate_model(model, test_loader):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)

            if isinstance(model, models.Inception3):
                if hasattr(outputs, "logits"):
                    outputs = outputs.logits

            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return acc, f1

def save_model_and_get_size_mb(model, file_name):
    torch.save(model.state_dict(), file_name)
    size_mb = os.path.getsize(file_name) / (1024 * 1024)
    return size_mb

def measure_inference_time_ms(model, dataset, n_samples=100):
    model.eval()
    n_samples = min(n_samples, len(dataset))
    indices = np.random.choice(len(dataset), n_samples, replace=False)

    times = []
    with torch.no_grad():
        for idx in indices:
            img, _ = dataset[idx]
            img = img.unsqueeze(0).to(device)

            if device.type == "cuda":
                torch.cuda.synchronize()

            start = time.time()
            outputs = model(img)

            if isinstance(model, models.Inception3):
                if hasattr(outputs, "logits"):
                    outputs = outputs.logits

            if device.type == "cuda":
                torch.cuda.synchronize()

            end = time.time()
            times.append((end - start) * 1000)

    return np.mean(times)

results = []

models_to_evaluate = [
    ("ResNet50", resnet_model, "resnet50_mango.pth"),
    ("InceptionV3", inception_model, "inceptionv3_mango.pth"),
    ("MobileNetV2", mobilenet_model, "mobilenetv2_mango.pth"),
]

for model_name, model, file_name in models_to_evaluate:
    acc, f1 = evaluate_model(model, test_loader)
    size_mb = save_model_and_get_size_mb(model, file_name)
    inference_ms = measure_inference_time_ms(model, test_dataset, n_samples=100)

    results.append({
        "Modelo": model_name,
        "Accuracy (test)": round(acc, 4),
        "F1-Score Macro (test)": round(f1, 4),
        "Tamaño modelo (MB)": round(size_mb, 2),
        "Tiempo inferencia (ms/img)": round(inference_ms, 2)
    })

results_df = pd.DataFrame(results).sort_values(by="F1-Score Macro (test)", ascending=False)

print("\nResultados finales:")
print(results_df)

results_df.to_csv("resultados_finales_mango.csv", index=False)
print("\nResultados guardados en: resultados_finales_mango.csv")


Resultados finales:
        Modelo  Accuracy (test)  F1-Score Macro (test)  Tamaño modelo (MB)  \
0     ResNet50            1.000                 1.0000               90.04   
2  MobileNetV2            0.995                 0.9950                8.76   
1  InceptionV3            0.915                 0.9155               93.28   

   Tiempo inferencia (ms/img)  
0                       19.85  
2                        5.72  
1                       17.17  

Resultados guardados en: resultados_finales_mango.csv
